In [1]:
#Set project root variables
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)

In [2]:

tables = pd.read_sql_query(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""",
conn
)

tables

,name
0,demo
1,drug
2,faers_severity_dataset
3,fluorouracil_filtered
4,indi
5,outc
6,reac
7,rpsr
8,ther


In [3]:

like_5FU_tables = pd.read_sql_query("""

SELECT DISTINCT drugname
FROM drug
WHERE drugname LIKE '%FLUOROURACIL%'
ORDER BY drugname;
        """,
        conn)

like_5FU_tables                   


,drugname
0,5-FLUOROURACIL /00098801/
1,5fluorouracile
2,CALCIPOTRIENE\FLUOROURACIL
3,CARBOPLATIN;FLUOROURACIL
4,CYCLOPHOSPHAMIDE\EPIRUBICIN HYDROCHLORIDE\FLUO...
5,EPIRUBICIN;FLUOROURACIL
6,FLUOROURACIL
7,"FLUOROURACIL ACCORD 50 mg/ml, solution for inf..."
8,FLUOROURACIL ANABIOSIS
9,FLUOROURACIL CO [FLUOROURACIL]


In [4]:

tables = pd.read_sql_query(
"""
SELECT drugname, COUNT(*) AS reports
FROM drug
WHERE drugname LIKE '%FLUOROURACIL%'
ORDER BY drugname;
""",
conn
)

tables

,drugname,reports
0,FLUOROURACIL,13254


In [ ]:
number_of_diff_FU = pd.read_sql_query("""SELECT
    drugname,
    COUNT(*) AS report_count
FROM drug
WHERE drugname LIKE '%FLUOROURACIL%'
GROUP BY drugname
ORDER BY report_count DESC;""", conn)

number_of_diff_FU

,drugname,report_count
0,FLUOROURACIL,12625
1,FLUOROURACIL\LEUCOVORIN CALCIUM\OXALIPLATIN,265
2,FLUOROURACIL\IRINOTECAN\LEUCOVORIN,124
3,FLUOROURACIL SODIUM,76
4,FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN,55
5,FLUOROURACIL\LEUCOVORIN,29
6,CYCLOPHOSPHAMIDE\EPIRUBICIN HYDROCHLORIDE\FLUO...,25
7,FLUOROURACIL\LEUCOVORIN\OXALIPLATIN,12
8,TOLAK [FLUOROURACIL],4
9,FLUOROURACIL\IRINOTECAN,3


In [ ]:
conn.execute("DROP TABLE IF EXISTS fluorouracil_filtered;")
conn.execute("""
CREATE TABLE fluorouracil_filtered AS
SELECT *
FROM drug
WHERE drugname IN (
    SELECT drugname
    FROM drug
    WHERE drugname LIKE '%FLUOROURACIL%'
    GROUP BY drugname
    HAVING COUNT(*) > 10
);
""")



In [ ]:
FU_df_filtered = pd.read_sql_query("""
SELECT *
FROM fluorouracil_filtered
""", conn)

FU_df_filtered.shape

(13211, 21)

In [ ]:
FU_df_filtered

,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,val_vbm,route,dose_vbm,cum_dose_chr,...,dechal,rechal,lot_num,exp_dt,nda_num,dose_amt,dose_unit,dose_form,dose_freq,source_quarter
0,113800407,11380040,24,SS,FLUOROURACIL,FLUOROURACIL,1,Unknown,"UNK, Cyclic",NaN,...,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q1
1,113800407,11380040,25,SS,FLUOROURACIL,FLUOROURACIL,1,Unknown,UNK,NaN,...,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q1
2,1177955113,11779551,38,C,FLUOROURACIL,FLUOROURACIL,1,NaN,UNK,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cream,NaN,Q1
3,125820364,12582036,8,SS,FLUOROURACIL,FLUOROURACIL,1,Unknown,cycle 8 day 1,NaN,...,U,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q1
4,125820364,12582036,9,SS,FLUOROURACIL,FLUOROURACIL,1,Unknown,UNK,NaN,...,U,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13206,247943981,24794398,2,SS,FLUOROURACIL,FLUOROURACIL,1,Unknown,9 cycles,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q4
13207,247950291,24795029,2,SS,FLUOROURACIL,FLUOROURACIL,1,Intravenous (not otherwise specified),NaN,NaN,...,U,NaN,UNKNOWN,NaN,NaN,4654.0,MG,NaN,NaN,Q4
13208,93587883,9358788,3,SS,FLUOROURACIL,FLUOROURACIL,1,Parenteral,"UNK UNK,QCY",NaN,...,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/CYCLE,Q4
13209,93587883,9358788,4,SS,FLUOROURACIL,FLUOROURACIL,1,NaN,NaN,NaN,...,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Q4
